# 10. AIND Ecephys NWB Export

Build an `AINDEcephysNWBScanConfig`, expand it with `GridScanGenerationTask`, and run the `AINDEcephysNWBTask` for each coordinate. The task itself clones [aind-ecephys-nwb](https://github.com/AllenNeuralDynamics/aind-ecephys-nwb) on first use, patches `run_capsule.py` for neuroconv API drift (`add_electrodes_info_to_nwbfile` → `add_electrodes_to_nwbfile`), seeds its `data/` with the dispatch `job_*.json`, runs `code/run_capsule.py`, and copies the resulting NWB file(s) into the single config's `coordinate_output_root` (`obi-output/10_aind_ecephys_nwb/grid_scan/0/`).

We `--write-raw` to include the raw `ElectricalSeries` and `--skip-lfp` (the toy data has no separate LFP stream and modern spikeinterface's `highpass_check` rejects the capsule's hard-coded LFP `freq_min=0.5`).

## Imports and prerequisites

In [1]:
import subprocess
import sys
from pathlib import Path

import obi_one as obi

In [2]:
subprocess.run(
    [
        "uv", "pip", "install", "--python", sys.executable,
        "neuroconv", "pynwb", "hdmf-zarr", "aind-nwb-utils",
    ],
    check=True,
)

Using Python 3.12.9 environment at: /Users/james/Documents/obi/code/obi-main/obi-one/.venv
Checked 4 packages in 11ms


CompletedProcess(args=['uv', 'pip', 'install', '--python', '/Users/james/Documents/obi/code/obi-main/obi-one/.venv/bin/python', 'neuroconv', 'pynwb', 'hdmf-zarr', 'aind-nwb-utils'], returncode=0)

## Build the scan config

In [3]:
dispatch_output_path = (
    Path.cwd() / "../../../obi-output/01_aind_ephys_dispatch/grid_scan/0"
).resolve()
assert dispatch_output_path.exists(), (
    f"{dispatch_output_path} not found. Run 01_aind_ephys_dispatch.ipynb first."
)

scan_config = obi.AINDEcephysNWBScanConfig(
    initialize=obi.AINDEcephysNWBScanConfig.Initialize(
        dispatch_output_path=dispatch_output_path,
        backend="hdf5",
        write_raw=True,
        skip_lfp=True,
    ),
)

## Generate the grid scan and run the NWB-export task

In [4]:
grid_scan = obi.GridScanGenerationTask(
    form=scan_config,
    output_root="../../../obi-output/10_aind_ecephys_nwb/grid_scan",
    coordinate_directory_option="ZERO_INDEX",
)
grid_scan.execute()

obi.run_tasks_for_generated_scan(grid_scan)

[2026-04-29 18:15:43,472] INFO: Seeded 1 job file(s) into /tmp/aind-ecephys-nwb/data


[2026-04-29 18:15:43,472] INFO: Running python -u run_capsule.py --backend hdf5 --lfp_temporal_factor 2 --lfp_spatial_factor 4 --lfp_highpass_freq_min 0.1 --write-raw --skip-lfp


[2026-04-29 18:15:47,484] INFO: 

NWB EXPORT ECEPHYS
Running NWB conversion with the following parameters:
Stub test: False
Stub seconds: 10.0
Write LFP: False
Write RAW: True
Temporal subsampling factor: 2
Spatial subsampling factor: 4
Highpass filter frequency: 0.1
Surface channel indices for agar probes: None
NWB backend: hdf5
Found 1 JSON job files
Session: session0
Number of NWB files to write: 1
Number of streams to write for each file: 1
Creating NWB file with info.
Processing block0_None_recording1
	Loading block0_None_recording1 from 1 JSON files
		block0_None_recording1
	Adding probe device: Probe from recording metadata
	Adding RAW data for stream None - segment 0
Added 1 streams
Configuring hdf5 backend
Writing NWB file to ../results/session0_block0_recording1.nwb
Writing time: 2.5s
Done writing ../results/session0_block0_recording1.nwb
NWB EXPORT ECEPHYS time: 2.55s



[PosixPath('../../../obi-output/10_aind_ecephys_nwb/grid_scan/0')]

## Open the NWB file with pynwb and inspect contents

In [5]:
from pynwb import NWBHDF5IO

coord_dir = Path(grid_scan.single_configs[0].coordinate_output_root)
print("coordinate_output_root:", coord_dir)

nwb_files = sorted(coord_dir.glob("*.nwb"))
print("NWB files:", [p.name for p in nwb_files])

if nwb_files:
    with NWBHDF5IO(str(nwb_files[0]), "r") as io:
        nwb = io.read()
        print("session_id:        ", nwb.session_id)
        print("identifier:        ", nwb.identifier)
        print("session_start_time:", nwb.session_start_time)
        print("devices:           ", list(nwb.devices.keys()))
        print("electrode_groups:  ", list(nwb.electrode_groups.keys()))
        if nwb.electrodes is not None:
            print("electrodes:        ", len(nwb.electrodes), "rows")
        print("acquisition:       ", list(nwb.acquisition.keys()))
        for name, obj in nwb.acquisition.items():
            shape = getattr(obj, "data", None).shape if hasattr(obj, "data") else "n/a"
            print(f"  {name}: shape={shape} rate={getattr(obj, 'rate', 'n/a')}")

coordinate_output_root: ../../../obi-output/10_aind_ecephys_nwb/grid_scan/0
NWB files: ['session0_block0_recording1.nwb']
session_id:         session0
identifier:         cf1be05e-c72d-4ac2-aaa1-9de43d372dae
session_start_time: 2026-04-29 18:15:44.776424+02:00
devices:            ['Probe']
electrode_groups:   ['Probe']
electrodes:         70 rows
acquisition:        ['ElectricalSeriesProbe']
  ElectricalSeriesProbe: shape=(300000, 70) rate=None
